In [ ]:
try:
    from google.colab import drive
    drive.mount('/gdrive')
    dataset_root = '/gdrive/MyDrive/datasets'
    !pip install torchinfo tqdm
    colab = True
except Exception as e:
    print(e)
    print('Assuming we\'re not on colab.')
    dataset_root = './datasets'
    colab = False

print('Will store datasets in', dataset_root)

import os

if os.name == 'nt':
    print("Disabling multiprocessing because we're running on windows.")
    cpu_num = 0
elif colab:
    cpu_num = 2
else:
    cpu_num = os.cpu_count() // 2
    print('Dataloaders will use {} CPUs'.format(cpu_num))

In [ ]:
import random

import torch
import torch.utils.data as tud
import torch.nn as nn
import torch.nn.functional as F

import torchvision.transforms.v2 as tv2
import torchvision.transforms.v2.functional as tvf
import torchvision.datasets as tds
import torchvision.utils as tu
import torchvision.ops as tvo
import torchvision

from torchinfo import summary
from tqdm import tqdm
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
voc_class_names = ["background", "aeroplane", "bicycle", "bird", "boat", "bottle",
                   "bus", "car", "cat", "chair", "cow", "diningtable", "dog", "horse",
                   "motorbike", "person", "pottedplant", "sheep", "sofa", "train", "tvmonitor"]
nn_dim = (320, 320)
train_tfs_v2 = tv2.Compose([
    tv2.ToImage(),
    tv2.RandomHorizontalFlip(0.5),
    # tv2.RandomResizedCrop(nn_dim, antialias=True),
    tv2.Resize(nn_dim, antialias=True),
    tv2.SanitizeBoundingBoxes(),
    tv2.ToDtype(torch.float32, scale=True),
    tv2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    tv2.ConvertBoundingBoxFormat('XYWH'),
])

val_tfs_v2 = tv2.Compose([
    tv2.ToImage(),
    tv2.Resize(nn_dim, antialias=True),
    tv2.SanitizeBoundingBoxes(),
    tv2.ToDtype(torch.float32, scale=True),
    tv2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    tv2.ConvertBoundingBoxFormat('XYWH'),
])

img_train = tds.VOCDetection(
    root=dataset_root,
    download=True,
    year='2012',
    image_set='train',
    transforms=train_tfs_v2
)
img_train = tds.wrap_dataset_for_transforms_v2(img_train)


img_val = tds.VOCDetection(
    root=dataset_root,
    download=True,
    year='2012',
    image_set='val',
    transforms=val_tfs_v2
)
img_val = tds.wrap_dataset_for_transforms_v2(img_val)

In [ ]:
def xywh_to_xyxy(boxes):
    boxes[..., 2] = boxes[..., 0] + boxes[..., 2]
    boxes[..., 3] = boxes[..., 1] + boxes[..., 3]
    return boxes

def xywh_to_cxywh(boxes):
    boxes[..., 0] = boxes[..., 0] + boxes[..., 2] // 2
    boxes[..., 1] = boxes[..., 1] + boxes[..., 3] // 2
    return boxes
    
class YoloConvertBoxes(nn.Module):
    def __init__(self, stride, canvas_size, num_classes):  
        super().__init__()
        self.stride = stride
        
        self.num_classes = num_classes
        
        self.w, self. h = canvas_size
        assert (self.w % stride == 0) and (self.h % stride == 0)
        
        w, h = canvas_size
        assert (w % stride == 0) and (h % stride == 0)
        
    # Convert boxes to be relative to the image cell
    # [cellx, celly, coffsetx, coffsety, width, height]
    def convert_xywh_to_cells(self, boxes):
        as_cxywh = xywh_to_cxywh(boxes.float())
        # print("CXY: ", as_cxywh)
        
        # nc = nearest_corners
        nc = torch.zeros_like(as_cxywh)
        nc[..., 0] = (as_cxywh[..., 0] // self.stride) * self.stride
        nc[..., 1] = (as_cxywh[..., 1] // self.stride) * self.stride
        cell_ix = (nc / self.stride)[..., :2].long()
        # print("Cell IX: ", cell_ix)
        # print("Nearest Corner: ", nc)
        
        # xoffset, yoffset, w, h
        offsets = (as_cxywh - nc)
        # print("Offsets: ", offsets)
        
        # Normalize the offset from nearest cell corner to the stride and
        # normalize the width and height to the canvas size. This places all
        # values between 0 and 1.
        offsets[..., 0] = offsets[..., 0] / self.stride
        offsets[..., 1] = offsets[..., 1] / self.stride
        offsets[..., 2] = offsets[..., 2] / self.w
        offsets[..., 3] = offsets[..., 3] / self.h
        
        return cell_ix, offsets
        
    
    # Boxes in [N, num_cells, 4]
    # * xywh ordering, x & y are the top left corner
    # * num_cells is (canvas_dim / stride)^2 for each stride.
    # Classes are onehot in [N, num_classes]
    def forward(self, tboxes, tclasses):
        cell_ixs, converted_tgt_box = self.convert_xywh_to_cells(tboxes)

        class_grid = torch.zeros(
            self.h // self.stride,
            self.w // self.stride,
            self.num_classes,
            dtype=torch.long
        )
        
        box_grid = torch.zeros(
            self.h // self.stride,
            self.w // self.stride,
            4,
            dtype=torch.float32
        )

        for ix, cl, orig_box, bb in zip(cell_ixs, tclasses, tboxes, converted_tgt_box):
            (y, x) = ix
            xcmin, ycmin, xcmax, ycmax = xywh_to_xyxy(orig_box) // stride
            class_grid[ycmin:ycmax+1, xcmin:xcmax+1, :] |= cl
            box_grid[y, x, :] = bb
        
        return class_grid.float(), box_grid
        

In [ ]:
class VocDetectionWrapper():
    def __init__(self, voc, stride):
        self.voc = voc
        self.stride = stride
        self.canvas_size = (320, 320)
        self.num_classes = 20
        
        self.converter = YoloConvertBoxes(self.stride, self.canvas_size, self.num_classes)
    
    def __getitem__(self, ix):
        img, boxes = self.voc[ix]
    
        boxes['boxes'].data = boxes['boxes']
        boxes['labels'] = F.one_hot(boxes['labels'].to(torch.long) - 1, num_classes=self.num_classes)
        # Returns boxes in xywh format normalized to [0,1] according to box dimensions
        # Returns the classes as a one hot vector.
        # return img, boxes
        
        cgrid, bgrid = self.converter(boxes['boxes'], boxes['labels'])
        out = torch.cat((bgrid, cgrid), axis=-1)
        # return img, out.flatten(0, 1)
        return img, cgrid, bgrid
    
    def __len__(self):
        return len(self.voc)
    
    def plot(self, ix, thresh=0.1):
        img, cgrid, bgrid = self.__getitem__(ix)
        
        gy, gx, _ = cgrid.shape
        w0, h0 = self.canvas_size
        
        labels = []
        boxes_to_plot = []
        for iy in range(gy):
            for ix in range(gx):
                if torch.sum(bgrid[iy, ix, :]) > 1e-6:
                # if torch.max(cgrid[iy, ix, :]) > thresh:
                    # labels.append(voc_class_names[(torch.argmax(cgrid[iy, ix, :]).item() + 1)])
                    
                    offsetx = bgrid[iy, ix, 0].item() * self.stride
                    offsety = bgrid[iy, ix, 1].item() * self.stride

                    
                    w = bgrid[iy, ix, 2].item() * w0
                    h = bgrid[iy, ix, 3].item() * h0

                    globalx = (iy * self.stride) + offsetx - w//2
                    globaly = (ix * self.stride) + offsety - h//2


                    # Now xyxy
                    boxes_to_plot.append(torch.tensor([globalx, globaly, globalx + w, globaly + h]))
        boxes_to_plot = torch.stack(boxes_to_plot)
        
        if img.dtype.is_floating_point and img.min() < 0:
            # Poor man's re-normalization for the colors to be OK-ish. This
            # is useful for images coming out of Normalize()
            img -= img.min()
            img /= img.max()

        img = tvf.to_dtype(img, torch.uint8, scale=True)
        drawn = tu.draw_bounding_boxes(
            img,
            boxes_to_plot)
            # labels=labels, width=4,
            # font="Ubuntu-M.ttf", font_size=32)
        plt.imshow(drawn.permute(1, 2, 0))

stride = 40
dset_train = VocDetectionWrapper(img_train, stride)
dset_val = VocDetectionWrapper(img_val, stride)

In [ ]:
i=102
img, cgrid, bgrid = dset_val[i]
# print("BGRID\n", bgrid)\
print("CGRID Person\n", cgrid[:, :, 14])
print("CGRID Table\n", cgrid[:, :, 10])
dset_val.plot(i)

In [ ]:
batchsize = 16

train_loader = tud.DataLoader(dset_train, batch_size=batchsize, num_workers=cpu_num, shuffle=True)
val_loader = tud.DataLoader(dset_val, num_workers=cpu_num, batch_size=batchsize, shuffle=True)

In [ ]:
class ShufflenetYolo(nn.Module):
    def __init__(self, stride=20, hiddensz=4096):
        super().__init__()
        self.stride = stride
        self.cell_dims = 320 // stride
        self.backbone = torchvision.models.shufflenet_v2_x1_0(weights='IMAGENET1K_V1')
        # self.conv_out = nn.Conv2d(1024, out_channels, kernel_size=1, padding=0, stride=1)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(p=0.5)
        self.fc0 = nn.Linear(1024*10*10, hiddensz)
        self.fc1 = nn.Linear(hiddensz, self.cell_dims * self.cell_dims * 24)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        # Copy paste from Shufflenet source code with final class head removed.
        x = self.backbone.conv1(x)
        x = self.backbone.maxpool(x)
        x = self.backbone.stage2(x)
        x = self.backbone.stage3(x)
        x = self.backbone.stage4(x)
        x = self.backbone.conv5(x)
        # x = x.mean([2, 3])  # globalpool
        x = self.fc0(x.flatten(start_dim=1))
        x = self.activation(x)
        x = self.dropout(x)
        x = self.fc1(x)
        x = x.reshape([-1, self.cell_dims, self.cell_dims, 24])
        x = self.sigmoid(x)
        return x
    
from torchvision.models._utils import IntermediateLayerGetter

class ResnetYolo(nn.Module):
    def __init__(self, stride=20, hiddensz=4096):
        super().__init__()
        self.stride = stride
        self.cell_dims = 320 // stride
        self.resnet = torchvision.models.resnet34(weights='IMAGENET1K_V1')
        self.backbone = IntermediateLayerGetter(self.resnet, return_layers={'layer4': "0"})

        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(p=0.5)
        self.fc0 = nn.Linear(512*10*10, hiddensz)
        self.fc1 = nn.Linear(hiddensz, self.cell_dims * self.cell_dims * 24)
        self.sigmoid = nn.Sigmoid()
        
    
    def forward(self, x):
        for name, layer in self.backbone.items():
            x = layer(x)

        x = self.fc0(x.flatten(start_dim=1))
        x = self.activation(x)
        x = self.dropout(x)
        x = self.fc1(x)
        x = x.reshape([-1, self.cell_dims, self.cell_dims, 24])
        x = self.sigmoid(x)
        return x
    
m = ResnetYolo(40)
summary(ResnetYolo(40), (1,3,320,320))

# import timeit
# m = Resnet18Yolo(stride).to(device)
# t = timeit.timeit(stmt='m(torch.randn(1,3,320,320, device=\'cuda\'))', globals=globals(), number=100)
# t / 100

In [ ]:
model = ResnetYolo(stride).to(device).train()

epochs = 100
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

class_lossfn = nn.BCELoss()
bbox_lossfn = nn.MSELoss()

c_loss_plot = []
b_loss_plot = []
for epoch in range(epochs):
    model.train()
    c_loss_avg = []
    b_loss_avg = []
    for i, (images, cgrid, bgrid) in enumerate(train_loader):
        optimizer.zero_grad()
        images = images.float().to(device)

        # p_cgrid = torch.randn(8, 16*16, 20).requires_grad_()
        # p_bgrid = torch.randn(8, 16*16, 4).requires_grad_()

        out = model(images)

        p_bgrid = out[..., :4].flatten(1,2)
        p_cgrid = out[..., 4:].flatten(1,2)
        
        ctgt = cgrid.to(device).flatten(1,2)
        btgt = bgrid.to(device).flatten(1,2)
        
        # Boxes with positive labels
        pos_boxes = (btgt.sum(dim=2) > 1e-5)
        bbox_loss = bbox_lossfn(p_bgrid[pos_boxes], btgt[pos_boxes])

        # Only penalize classes where there's at least one GT prediction
        pos_class = (ctgt.sum(dim=2) > 1e-5)
        class_loss = class_lossfn(p_cgrid[pos_class], ctgt[pos_class])

        # print(p_bgrid[pos_boxes], p_cgrid[pos_class])
        # print(btgt[pos_boxes], ctgt[pos_class])

        loss = bbox_loss + class_loss
        loss.backward()
        c_loss_avg.append(class_loss.item())
        b_loss_avg.append(bbox_loss.item())
        optimizer.step()
    
    epoch_closs = torch.tensor(c_loss_avg).mean().item()
    epoch_bloss = torch.tensor(b_loss_avg).mean().item()
    scheduler.step()
    c_loss_plot.append(epoch_closs)
    b_loss_plot.append(epoch_bloss)
    print(
        "Epoch: ", epoch,
        "LR: ", scheduler.get_last_lr(),
        "Class Loss", epoch_closs, 
        "Bbox Loss", epoch_bloss
    )

    

In [ ]:
fig, (ax1, ax2) = plt.subplots(1,2)
ax1.plot(b_loss_plot)
ax2.plot(c_loss_plot)

In [ ]:
def eval_plot(ix, dset, model, thresh=0.1):
    stride = 40
    model.eval()
    
    img, cgrid, bgrid = dset[ix]
    
    out = model(img.float().to(device).unsqueeze(0))

    p_bgrid = out[..., :4].squeeze(0)
    p_cgrid = out[..., 4:].squeeze(0)

    assert(cgrid.shape == p_cgrid.shape)
    assert(bgrid.shape == p_bgrid.shape)
    gy, gx, _ = cgrid.shape
    w0, h0 = (320, 320)

    labels = []
    boxes_to_plot = []
    for iy in range(gy):
        for ix in range(gx):
            
            # To check print only boxes that were assigned to objects in the GT.
            if torch.sum(bgrid[iy, ix, :]) > 1e-6:
                print("Pred: ",
                      ix,
                      iy,
                      voc_class_names[torch.argmax(p_cgrid[iy, ix, :]).item() + 1],
                      "\n",
                      p_cgrid[iy, ix, :]
                )
                print("GT: ",
                      ix, 
                      iy,
                      voc_class_names[torch.argmax(cgrid[iy, ix, :]).item() + 1],
                      "\n",
                      cgrid[iy, ix, :]
                )

                offsetx = p_bgrid[iy, ix, 0].item() * stride
                offsety = p_bgrid[iy, ix, 1].item() * stride


                w = p_bgrid[iy, ix, 2].item() * w0
                h = p_bgrid[iy, ix, 3].item() * h0

                globalx = (iy * stride) + offsetx - w//2
                globaly = (ix * stride) + offsety - h//2


                # Now xyxy
                boxes_to_plot.append(torch.tensor([globalx, globaly, globalx + w, globaly + h]))
    boxes_to_plot = torch.stack(boxes_to_plot)

    if img.dtype.is_floating_point and img.min() < 0:
        img -= img.min()
        img /= img.max()

    img = tvf.to_dtype(img, torch.uint8, scale=True)
    drawn = tu.draw_bounding_boxes(
        img,
        boxes_to_plot)
    plt.imshow(drawn.permute(1, 2, 0))

In [ ]:
from ipywidgets import interact

@interact(index=(0, len(dset_val) - 1, 1))
def draw_preds(index=0):
    with torch.no_grad():
        eval_plot(index, dset_val, model)